# Extracción de métricas acústicas y lingüísticas

Este notebook está basado en el enfoque descrito en el artículo [PMC9056005](https://pmc.ncbi.nlm.nih.gov/articles/PMC9056005/), adaptado para analizar chunks de audio y sus correspondientes transcripciones en un contexto de clasificación de afasia.

## Métricas Extraídas

Las siguientes métricas se generan a partir de los audios y transcripciones procesados:

### **Métricas Acústicas (OpenSMILE eGeMAPS)**
- `F0semitoneFrom27.5Hz_sma3nz_amean`
- `F0semitoneFrom27.5Hz_sma3nz_stddevNorm`
- `F0semitoneFrom27.5Hz_sma3nz_percentile20.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile50.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile80.0`
- `F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2`
- `F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope`
- `loudness_sma3_amean`
- `loudness_sma3_stddevNorm`
- `spectralFlux_sma3_amean`
- `spectralFlux_sma3_stddevNorm`
- `mfcc1_sma3_amean` a `mfcc4_sma3_stddevNorm`
- `jitterLocal_sma3nz_amean`
- `jitterLocal_sma3nz_stddevNorm`
- `shimmerLocaldB_sma3nz_amean`
- `shimmerLocaldB_sma3nz_stddevNorm`
- `HNRdBACF_sma3nz_amean`
- `HNRdBACF_sma3nz_stddevNorm`
- `alphaRatioV_sma3nz_amean`
- `alphaRatioV_sma3nz_stddevNorm`
- `hammarbergIndexV_sma3nz_amean`
- `hammarbergIndexV_sma3nz_stddevNorm`

### **Métricas Lingüísticas**
- Número total de palabras (`num_palabras`).
- Número de palabras únicas (`num_palabras_unicas`).
- Diversidad léxica (`diversidad_lexica`): proporción de palabras únicas respecto al total.
- Embeddings de BERT para las transcripciones.

### **Métricas Derivadas (Librosa)**
- `mfcc1_mean` a `mfcc13_mean`: Medias de los coeficientes MFCC.
- `mfcc1_stddev` a `mfcc13_stddev`: Desviaciones estándar de los coeficientes MFCC.

Estas métricas se combinan con las columnas originales del dataset para formar un único conjunto de datos listo para la modelización.

In [22]:
import librosa
import numpy as np
import pandas as pd
from opensmile import Smile,FeatureSet, FeatureLevel
from transformers import BertTokenizer, BertModel
import spacy

In [23]:
# Inicialización de librerías necesarias
nlp = spacy.load("es_core_news_sm")  # Para métricas lingüísticas
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # BERT embeddings
model = BertModel.from_pretrained("bert-base-uncased")

# Inicializar OpenSMILE para características acústicas específicas
smile_egemaps = Smile(feature_set=FeatureSet.eGeMAPSv02, feature_level=FeatureLevel.Functionals)

# Función para extraer características acústicas con librosa
def extraer_mfcc(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta_mfcc = librosa.feature.delta(mfccs)
    delta2_mfcc = librosa.feature.delta(mfccs, order=2)
    
    # Calcular estadísticas resumen
    features = {
        f"mfcc{i+1}_mean": np.mean(mfcc) for i, mfcc in enumerate(mfccs)
    }
    features.update({
        f"mfcc{i+1}_stddev": np.std(mfcc) for i, mfcc in enumerate(mfccs)
    })
    return features

# Función para extraer características lingüísticas
def extraer_linguisticas(transcripcion):
    # Procesamiento de texto con spaCy
    doc = nlp(transcripcion)
    num_palabras = len(doc)
    num_palabras_unicas = len(set([token.text for token in doc]))
    diversidad_lexica = num_palabras_unicas / num_palabras if num_palabras > 0 else 0
    
    # Embeddings de BERT
    inputs = tokenizer(transcripcion, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    bert_sent_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy().flatten()
    
    return {
        "num_palabras": num_palabras,
        "num_palabras_unicas": num_palabras_unicas,
        "diversidad_lexica": diversidad_lexica,
        "bert_embedding": bert_sent_embedding
    }

# Función principal para extraer TODAS las métricas para un chunk
def extraer_metricas_chunk(ruta_audio, transcripcion):
    try:
        # Cargar audio
        y, sr = librosa.load(ruta_audio, sr=None)

        # Características acústicas con librosa (MFCCs)
        mfcc_features = extraer_mfcc(y, sr)

        # Características acústicas con openSMILE
        egemaps_features = smile_egemaps.process_file(ruta_audio).to_dict(orient='records')[0]

        # Características lingüísticas
        linguistics = extraer_linguisticas(transcripcion)

        # Fusionar todas las métricas
        all_features = {
            **mfcc_features,
            **egemaps_features,
            **linguistics  # Incluye características lingüísticas
        }
        return all_features
    except Exception as e:
        print(f"Error procesando el audio {ruta_audio}: {e}")
        return None

In [24]:
path_data = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df = pd.read_csv(path_data + 'df_transcrip_chunk_info.csv', encoding='utf-8')

resultados = []

# Procesar cada fila del dataset
for idx, row in df.iterrows():
    ruta_audio = row['name_chunk_audio_path']
    transcripcion = row['Marca']  # Columna con las transcripciones

    # Extraer métricas para cada chunk
    metricas = extraer_metricas_chunk(ruta_audio, transcripcion)

    if metricas:
        resultado = row.to_dict()  # Copia todas las columnas originales
        resultado.update(metricas)  # Añade las nuevas métricas
        resultados.append(resultado)

# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados)

In [25]:
df_resultados.head()

,Inicio,Fin,Marca,Transcrip_name,Duración,name_chunk_audio,name_chunk_audio_path,CIP,NumId,Gènere,...,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp,diversidad_lexica,bert_embedding
0,0.00000,2.32800,IL,02_008_CAT_Conversacion,2.32800,02_008_CAT_Conversacion_0.00000_2.32800.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_008,8,1,...,0.035573,3.797468,2.192982,0.296000,0.197949,0.170000,0.140890,-45.265697,1.0,"[-0.25935647, -0.08536441, -0.3236561, -0.1502..."
1,0.41390,4.65332,n'hi ha una casa en un cotxe en la porta,02_007_CAT_Lamina,4.23942,02_007_CAT_Lamina_0.41390_4.65332.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_007,7,2,...,1.582209,3.294118,2.631579,0.296364,0.142973,0.077778,0.052446,-10.714849,0.9,"[-0.19006087, -0.20550142, 0.11232989, 0.26420..."
2,0.94175,2.84395,vale,02_002_CAT_Lamina,1.90220,02_002_CAT_Lamina_0.94175_2.84395.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_002,2,2,...,1.179943,1.075269,2.222222,0.162500,0.143592,0.267500,0.133112,-20.574884,1.0,"[0.015197615, 0.012435779, -0.23626824, -0.225..."
3,2.40829,3.47078,cotxe,02_007_CAT_Lamina,1.06249,02_007_CAT_Lamina_2.40829_3.47078.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_007,7,2,...,4.689577,4.950495,3.157895,0.250000,0.173781,0.046667,0.020548,-9.989378,1.0,"[0.009545505, -0.12974696, 0.119702145, 0.0275..."
4,2.69463,6.79340,ahora IL mi filla esta,02_008_CAT_Conversacion,4.09877,02_008_CAT_Conversacion_2.69463_6.79340.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_008,8,1,...,0.044272,2.450980,1.745636,0.364286,0.300707,0.187143,0.146747,-38.161377,1.0,"[-0.33293092, 0.023252353, -0.06884192, -0.022..."


In [26]:
df.columns

Index(['Inicio', 'Fin', 'Marca', 'Transcrip_name', 'Duración',
       'name_chunk_audio', 'name_chunk_audio_path', 'CIP', 'NumId', 'Gènere',
       'TipusAfàsia', 'LLengWAB', 'Edat', 'Grup', 'QA', 'Fluente/No Fluente',
       'num_palabras', 'num_palabras_unicas', 'promedio_palabras_por_frase',
       'num_ininteligibles', 'palabras_por_minuto', 'palabras_por_segundo'],
      dtype='object')

In [27]:
df_resultados.columns

Index(['Inicio', 'Fin', 'Marca', 'Transcrip_name', 'Duración',
       'name_chunk_audio', 'name_chunk_audio_path', 'CIP', 'NumId', 'Gènere',
       ...
       'spectralFluxUV_sma3nz_amean', 'loudnessPeaksPerSec',
       'VoicedSegmentsPerSec', 'MeanVoicedSegmentLengthSec',
       'StddevVoicedSegmentLengthSec', 'MeanUnvoicedSegmentLength',
       'StddevUnvoicedSegmentLength', 'equivalentSoundLevel_dBp',
       'diversidad_lexica', 'bert_embedding'],
      dtype='object', length=138)

In [29]:
ruta_base = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df.to_csv(ruta_base + 'df_transcrip_audio_metrics.csv', index=False, encoding='utf-8')